In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = "../results_flexlora_comparison"

def load(method, seed):
    path = os.path.join(RESULTS_DIR, f"{method}_seed{seed}_alpha00.json")
    if not os.path.exists(path):
        print(f"Missing: {path}")
        return None
    with open(path) as f:
        return json.load(f)

METHODS = ["flexlora", "spa_m", "homo_r8"]
SEEDS   = [42, 43, 44]
COLORS  = {"flexlora": "#2196F3", "spa_m": "#E91E63", "homo_r8": "#4CAF50"}
LABELS  = {"flexlora": "FlexLoRA", "spa_m": "SPA-M (Ours)", "homo_r8": "FedAvg r=8"}

# Load all available results
data = {}
for method in METHODS:
    for seed in SEEDS:
        d = load(method, seed)
        if d:
            data[(method, seed)] = d

print(f"Loaded {len(data)} runs")
for k in data:
    print(f"  {k[0]} seed={k[1]}: {len(data[k]['rounds'])} rounds")

In [ ]:
# Per-seed ROUGE-L curves
available_seeds = sorted(set(s for (m, s) in data.keys()))

fig, axes = plt.subplots(1, len(available_seeds), figsize=(6 * len(available_seeds), 4), sharey=True)
if len(available_seeds) == 1:
    axes = [axes]

for ax, seed in zip(axes, available_seeds):
    for method in METHODS:
        if (method, seed) not in data:
            continue
        rounds = data[(method, seed)]["rounds"]
        x = [r["round"] for r in rounds]
        y = [r["rouge_l_unseen"] for r in rounds]
        ax.plot(x, y, label=LABELS[method], color=COLORS[method], linewidth=2, marker="o", markersize=3)
    ax.set_title(f"Seed {seed}", fontsize=12)
    ax.set_xlabel("Round")
    ax.set_ylabel("ROUGE-L (unseen clients)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

fig.suptitle("FlexLoRA Metric Comparison — ROUGE-L on Unseen Clients\nQwen2.5-1.5B | Dolly-15K | Task-Heterogeneous", fontsize=13)
plt.tight_layout()
plt.savefig("../notebooks/flexlora_comparison_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table: Mean-L5, Final, Best per method
print(f"{'Method':<15} {'Seed':<6} {'Mean-L5':>10} {'Final':>10} {'Best':>10} {'Loss-Final':>12}")
print("-" * 65)

summary = {m: {"mean_l5": [], "final": [], "best": []} for m in METHODS}

for method in METHODS:
    for seed in SEEDS:
        if (method, seed) not in data:
            continue
        rounds = data[(method, seed)]["rounds"]
        rouge = [r["rouge_l_unseen"] for r in rounds]
        loss  = [r["avg_loss"] for r in rounds]
        ml5   = float(np.mean(rouge[-5:]))
        final = rouge[-1]
        best  = max(rouge)
        summary[method]["mean_l5"].append(ml5)
        summary[method]["final"].append(final)
        summary[method]["best"].append(best)
        print(f"{method:<15} {seed:<6} {ml5:>10.4f} {final:>10.4f} {best:>10.4f} {loss[-1]:>12.4f}")
    print()

print("=" * 65)
print(f"{'Method':<15} {'n':<6} {'Mean-L5 ★':>10} {'Final':>10} {'Best':>10}")
print("-" * 65)
for method in METHODS:
    ml5s = summary[method]["mean_l5"]
    fins = summary[method]["final"]
    bsts = summary[method]["best"]
    if not ml5s:
        continue
    n = len(ml5s)
    print(f"{method:<15} {n:<6} "
          f"{np.mean(ml5s):>6.4f}±{np.std(ml5s):.4f} "
          f"{np.mean(fins):>6.4f}±{np.std(fins):.4f} "
          f"{np.mean(bsts):>6.4f}±{np.std(bsts):.4f}")

In [ ]:
# Mean ± std curves across seeds
methods_with_data = [m for m in METHODS if summary[m]["mean_l5"]]

# Build per-round arrays
all_rouge = {m: [] for m in methods_with_data}
for method in methods_with_data:
    for seed in SEEDS:
        if (method, seed) not in data:
            continue
        rouge = [r["rouge_l_unseen"] for r in data[(method, seed)]["rounds"]]
        all_rouge[method].append(rouge)

fig, ax = plt.subplots(figsize=(9, 5))
for method in methods_with_data:
    arr = np.array(all_rouge[method])   # (n_seeds, n_rounds)
    mu  = arr.mean(axis=0)
    std = arr.std(axis=0)
    x   = np.arange(1, len(mu) + 1)
    ax.plot(x, mu, label=f"{LABELS[method]} (n={len(arr)})",
            color=COLORS[method], linewidth=2)
    ax.fill_between(x, mu - std, mu + std, alpha=0.15, color=COLORS[method])

ax.set_xlabel("Communication Round", fontsize=12)
ax.set_ylabel("ROUGE-L (unseen clients)", fontsize=12)
ax.set_title("SPA-M vs FlexLoRA — Zero-Shot Generalization to Unseen Clients\n"
             "Qwen2.5-1.5B | Dolly-15K | Task-Heterogeneous FL", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../notebooks/flexlora_comparison_mean_std.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Quick single-seed raw view (useful while only seed 42 is done)
seed = 42
print(f"=== Seed {seed} round-by-round ===")
print(f"{'Round':<6} ", end="")
for m in METHODS:
    if (m, seed) in data:
        print(f"{LABELS[m]:>15}", end="")
print()
print("-" * (6 + 15 * sum(1 for m in METHODS if (m, seed) in data)))

max_rounds = max(len(data[(m, seed)]["rounds"]) for m in METHODS if (m, seed) in data)
for i in range(max_rounds):
    print(f"R{i+1:<5}", end="")
    for m in METHODS:
        if (m, seed) not in data:
            continue
        rounds = data[(m, seed)]["rounds"]
        val = rounds[i]["rouge_l_unseen"] if i < len(rounds) else float("nan")
        print(f"{val:>15.4f}", end="")
    print()